# Урок 21 — Фінальний Проект: Mini E-commerce Analytics Engine

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Що ми будуємо?

У нас є **1 мільйон рядків** реальних транзакцій з Amazon India.
Аналітичний відділ хоче щоденний звіт із KPI:

- Які категорії найприбутковіші?
- Яка ставка повернень по кожній категорії?
- Які продавці мають найвищий рейтинг і найбільшу виручку?

Ми будуємо аналітичний рушій **з нуля** — від хаосу до системи.

| Частина | Концепція | Що будуємо |
|---------|-----------|------------|
| 1 | Задача + Дані | Завантаження 1M рядків e-commerce |
| 2 | Функції як аналітичні одиниці | 5 метрик через чисті функції |
| 3 | Функції як об'єкти | Пайплайн трансформацій |
| 4 | Декоратори | `@timer`, `@logged` |
| 5 | Перехід до класів | `DatasetAnalyzer` v1 |
| 6 | OOP: `@property`, дандери | `DatasetAnalyzer` v2 |
| 7 | Наслідування | `SalesAnalyzer`, `ReturnAnalyzer`, `SellerAnalyzer` |
| 8 | Композиція | `AnalyticsEngine` (Facade Pattern) |
| 9 | Контекстний менеджер | `DataLoader` |
| 10 | Streamlit | Мінімальний дашборд |

---

> **Головна ідея:** Функції → Класи → Система — це не просто синтаксис,
> а **природна еволюція складності**. Починаємо з хаосу. Закінчуємо продуктом.

## Частина 1 — Задача та Дані

---

### Структура датасету `amazon_ecommerce_1M.csv`

| Колонка | Тип | Опис |
|---------|-----|------|
| `user_id` | str | ID покупця |
| `product_id` | str | ID товару |
| `category` | str | Категорія: Electronics, Clothing, Home, Sports, Beauty |
| `subcategory` | str | Підкатегорія |
| `brand` | str | Бренд |
| `price` | float | Ціна до знижки |
| `discount` | float | Відсоток знижки |
| `final_price` | float | Фінальна ціна (після знижки) |
| `rating` | float | Оцінка покупця 1–5 |
| `review_count` | int | Кількість відгуків |
| `seller_id` | str | ID продавця |
| `seller_rating` | float | Рейтинг продавця |
| `purchase_date` | date | Дата замовлення |
| `shipping_time_days` | int | Час доставки (днів) |
| `location` | str | Місто покупця |
| `payment_method` | str | Спосіб оплати |
| `is_returned` | bool | Чи повернуто товар |
| `delivery_status` | str | Delivered / In Transit / Delayed / Returned |

### KPI, які ми будемо обчислювати

| KPI | Формула |
|-----|---------|
| Виручка | `sum(final_price)` для Delivered |
| Середній чек | `mean(final_price)` для Delivered |
| Ставка повернень | `mean(is_returned)` |
| Середня оцінка | `mean(rating)` |
| Середня доставка | `mean(shipping_time_days)` для Delivered |

In [ ]:
# Завантажуємо датасет
# 1M рядків — для навчання беремо перші 200k (швидше, але репрезентативно)
import pandas as pd
import numpy as np
from pathlib import Path

DATA_FILE = Path('amazon_ecommerce_1M.csv')
SAMPLE_SIZE = 200_000  # збільш до None щоб завантажити всі 1M рядків

print(f'Читаємо: {DATA_FILE} ...')
df = pd.read_csv(DATA_FILE, parse_dates=['purchase_date'], nrows=SAMPLE_SIZE)
print(f'Завантажено: {len(df):,} рядків × {len(df.columns)} стовпців')
print(f'Память: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

In [ ]:
# Перший погляд: що за дані нам дісталися?
print('=== Типи стовпців ===')
print(df.dtypes)
print()

display(df.head(3))

print('\n=== Статуси доставки ===')
print(df['delivery_status'].value_counts())

print('\n=== Категорії ===')
print(df['category'].value_counts())

print('\n=== Числова статистика ===')
display(df[['price', 'final_price', 'discount', 'shipping_time_days', 'rating']].describe().round(2))

## Частина 2 — Функції як Аналітичні Одиниці

---

### Ідея: одна функція = один KPI

Кожен бізнес-показник — це окрема, **чиста функція** (pure function):
- отримує DataFrame
- повертає одне число
- не змінює вхідні дані

**Чому це добре?**
- Тестується окремо: `assert return_rate(test_df) == 0.1`
- Передається як аргумент у функції вищого порядку
- Замінюється без зміни інших частин системи

```
df  ──▶  avg_order_value(df)  ──▶  число
         return_rate(df)      ──▶  число
         total_revenue(df)    ──▶  число
```

In [ ]:
# П'ять чистих функцій-метрик
# Кожна: DataFrame → одне число

def avg_order_value(df):
    """Середній чек — тільки по доставлених замовленнях."""
    delivered = df[df['delivery_status'] == 'Delivered']
    return round(delivered['final_price'].mean(), 2)


def return_rate(df):
    """Частка повернутих товарів (is_returned == True)."""
    return round(df['is_returned'].mean(), 4)


def total_revenue(df):
    """Загальна виручка (тільки Delivered)."""
    delivered = df[df['delivery_status'] == 'Delivered']
    return round(delivered['final_price'].sum(), 2)


def avg_delivery_days(df):
    """Середній час доставки (тільки Delivered)."""
    delivered = df[df['delivery_status'] == 'Delivered']
    return round(delivered['shipping_time_days'].mean(), 2)


def avg_customer_rating(df):
    """Середня оцінка покупців."""
    return round(df['rating'].mean(), 2)


# Тестуємо кожну функцію окремо
print(f'Середній чек:       ₹{avg_order_value(df):>12,.2f}')
print(f'Ставка повернень:    {return_rate(df):>11.1%}')
print(f'Загальна виручка:   ₹{total_revenue(df):>12,.0f}')
print(f'Середня доставка:    {avg_delivery_days(df):>10.1f} днів')
print(f'Середня оцінка:      {avg_customer_rating(df):>10.2f} / 5.0')

In [ ]:
# Функції в списку — обчислюємо всі KPI одним викликом
# Ключова ідея уроку 13: функція = об'єкт, який можна покласти в список

METRICS = [
    ('Середній чек',        avg_order_value),
    ('Ставка повернень',    return_rate),
    ('Загальна виручка',    total_revenue),
    ('Середня доставка',    avg_delivery_days),
    ('Середня оцінка',      avg_customer_rating),
]

# run_metrics — функція вищого порядку: приймає список функцій як аргумент
def run_metrics(df, metrics):
    """Запускає список (назва, функція) і повертає словник результатів."""
    return {name: func(df) for name, func in metrics}


results = run_metrics(df, METRICS)
for name, value in results.items():
    print(f'  {name:<25} {value}')

## Частина 3 — Функції як Об'єкти: Пайплайн Трансформацій

---

### Ідея: аналіз = конвеєр (pipeline)

Замість того щоб один раз обробити датафрейм і втратити проміжні кроки,
ми будуємо **декларативний пайплайн** — список трансформацій.

```
df  ──▶  filter_delivered  ──▶  add_revenue_band  ──▶  add_is_late  ──▶  df_enriched
```

Кожна функція: `DataFrame → DataFrame` (незмінна сигнатура = їх можна комбінувати).

Це патерн **Unix pipes**: `cat file | grep pattern | sort | head -10`

In [ ]:
# Функції-трансформації: DataFrame → DataFrame (збагачують датасет)

def filter_delivered(df):
    """Залишаємо тільки доставлені замовлення."""
    return df[df['delivery_status'] == 'Delivered'].copy()


def add_revenue_band(df):
    """Категорія замовлення за сумою: Low / Mid / High (в рупіях)."""
    df = df.copy()
    df['revenue_band'] = pd.cut(
        df['final_price'],
        bins=[0, 2000, 15000, float('inf')],
        labels=['Low', 'Mid', 'High']
    )
    return df


def add_is_late(df, threshold=7):
    """Позначаємо 'запізнілі' доставки (> threshold днів)."""
    df = df.copy()
    df['is_late'] = df['shipping_time_days'] > threshold
    return df


def add_discount_tier(df):
    """Категорія знижки: None (0%) / Small (1-20%) / Big (>20%)."""
    df = df.copy()
    df['discount_tier'] = pd.cut(
        df['discount'],
        bins=[-1, 0, 20, 100],
        labels=['None', 'Small', 'Big']
    )
    return df


# Пайплайн — просто список функцій
PIPELINE = [filter_delivered, add_revenue_band, add_is_late, add_discount_tier]


def run_pipeline(df, pipeline):
    """Послідовно застосовує кожну функцію пайплайну до DataFrame."""
    result = df
    for step in pipeline:
        result = step(result)
        print(f'  ✓ {step.__name__:<25} → {len(result):>7,} рядків')
    return result


print('Запускаємо пайплайн:')
df_enriched = run_pipeline(df, PIPELINE)
print()
df_enriched[['final_price', 'revenue_band', 'shipping_time_days', 'is_late', 'discount_tier']].head(5)

## Частина 4 — Декоратори: Прозора інфраструктура

---

### Ідея: Cross-cutting concerns

У нас є кілька функцій аналізу. Кожна повинна:

* вимірювати час виконання
* логувати виклик і результат

---

### Два підходи

* **Імперативний підхід**: логіка логування додається безпосередньо в кожну функцію
* **Декларативний підхід**: логіка винесена в окремий декоратор і застосовується зверху

---

> `@timer` — це скорочення для `func = timer(func)`
> Декоратор обгортає функцію, не змінюючи її бізнес-логіку.

---

### Порівняння

| Імперативний підхід                    | Декларативний підхід             |
| -------------------------------------- | -------------------------------- |
| Копіюємо `time.time()` у кожну функцію | Пишемо один `@timer`             |
| Логіка розмазана по коду               | Логіка централізована            |
| Зміна → правка багатьох місць          | Зміна → правка одного декоратора |

---

## 🧠 Інсайт

> Декоратори дозволяють винести **інфраструктурну логіку**
> (логування, метрики, кешування) за межі бізнес-коду

---


In [ ]:
import time
import functools


def timer(func):
    """Вимірює і виводить час виконання функції."""
    @functools.wraps(func)  # зберігає __name__, __doc__ оригіналу
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = (time.perf_counter() - start) * 1000
        print(f'[TIMER] {func.__name__:<30} {elapsed:.1f} ms')
        return result
    return wrapper


def logged(func):
    """Логує розмір вхідного df та результат."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        n_rows = len(args[0]) if args else '?'
        print(f'[LOG]   {func.__name__}(df з {n_rows:,} рядків) ...')
        result = func(*args, **kwargs)
        print(f'[LOG]   → {result}')
        return result
    return wrapper


# Декоруємо наші функції-метрики
@timer
@logged
def avg_order_value_v2(df):
    """Середній чек (Delivered)."""
    delivered = df[df['delivery_status'] == 'Delivered']
    return round(delivered['final_price'].mean(), 2)


@timer
def return_rate_v2(df):
    """Ставка повернень."""
    return round(df['is_returned'].mean(), 4)


print('=== Виклик з декораторами ===')
avg_order_value_v2(df)
print()
return_rate_v2(df)

## Частина 5 — Перехід до Класів: Від Хаосу до Структури

---

### Проблема: 15+ вільних функцій без організації

Уявіть: є 15 функцій-метрик, 4 трансформації, 2 декоратори.
Розробник запитує: *«Де метрики для продавців? Де фільтр по регіону?
Де кешований результат минулого запуску?»*

Відповіді немає — все плоске у глобальному просторі.

**Рішення:** клас `DatasetAnalyzer` — **іменований простір з пам'яттю**.

```python
# Без класу — хаос
delivered_df = df[df['delivery_status'] == 'Delivered']  # глобальна змінна
result_rev = total_revenue(df)  # де зберігається? хто знає

# З класом — порядок
analyzer = DatasetAnalyzer(df)
analyzer.total_revenue()        # метод = функція, прив'язана до стану
```

| Без класу | З класом |
|-----------|----------|
| 15 глобальних функцій | Методи в одному місці |
| Стан у глобальних змінних | Стан в `self` (захищений) |
| Складно передати «аналізатор» | Один об'єкт із усім потрібним |

In [ ]:
# DatasetAnalyzer v1 — базова структура

class DatasetAnalyzer:
    """Аналізатор e-commerce датасету."""

    def __init__(self, df: pd.DataFrame, name: str = 'Dataset'):
        # __init__ ініціалізує стан — зберігаємо в self
        self.df = df.copy()  # захист: копія, щоб не змінювати оригінал
        self.name = name
        self._results = {}   # _ означає: внутрішній, не для зовнішнього коду

    # ── Метрики (публічні методи) ──────────────────────────────────────────

    def avg_order_value(self):
        """Середній чек по доставлених."""
        delivered = self.df[self.df['delivery_status'] == 'Delivered']
        return round(delivered['final_price'].mean(), 2)

    def return_rate(self):
        """Частка повернень."""
        return round(self.df['is_returned'].mean(), 4)

    def total_revenue(self):
        """Загальна виручка (Delivered)."""
        delivered = self.df[self.df['delivery_status'] == 'Delivered']
        return round(delivered['final_price'].sum(), 2)

    def summary(self):
        """Виводить зведений звіт."""
        print(f'=== {self.name} — KPI Summary ===')
        print(f'  Записів:          {len(self.df):>10,}')
        print(f'  Виручка:          ₹{self.total_revenue():>12,.0f}')
        print(f'  Середній чек:     ₹{self.avg_order_value():>12,.2f}')
        print(f'  Ставка повернень:  {self.return_rate():>10.1%}')


# Використання
analyzer = DatasetAnalyzer(df, name='Amazon India — 200k')
analyzer.summary()

## Частина 6 — OOP: `@property`, Дандер-методи, Інкапсуляція

---


### Три речі, що роблять клас «дорослим»

**1. `@property`** — обчислюваний атрибут без `()`:
```python
analyzer.row_count   # замість analyzer.row_count()
analyzer.delivered   # ледачий фільтр — виглядає як поле
```

**2. `__repr__`** — текстове подання замість `<object at 0x7f...>`:
```python
print(analyzer)  # DatasetAnalyzer(name='Amazon', rows=200,000)
```

**3. `_приватні` методи** — внутрішня деталь реалізації:
```python
self._compute_metric(...)   # _ сигналізує: «не чіпай ззовні»
```

**4. Кеш** — обчислюємо метрику один раз, далі з кешу:
перший виклик → рахуємо; наступні → миттєво.

In [ ]:
# DatasetAnalyzer v2 — повна версія з @property, дандерами, кешем

class DatasetAnalyzer:
    """Аналізатор e-commerce датасету — повна версія."""

    def __init__(self, df: pd.DataFrame, name: str = 'Dataset'):
        self._df = df.copy()  # _df — захищений: зовнішній код не змінює напряму
        self._name = name
        self._cache = {}      # кеш результатів обчислень

    # ── @property — властивості ─────────────────────────────────────────────

    @property
    def name(self):
        """Назва аналізу (read-only)."""
        return self._name

    @property
    def row_count(self):
        """Кількість рядків — виглядає як поле, а не метод."""
        return len(self._df)

    @property
    def delivered(self):
        """Підмножина доставлених замовлень."""
        return self._df[self._df['delivery_status'] == 'Delivered']

    @property
    def returned(self):
        """Підмножина повернених товарів."""
        return self._df[self._df['is_returned']]

    @property
    def date_range(self):
        """Діапазон дат датасету у вигляді рядка."""
        mn = self._df['purchase_date'].min().date()
        mx = self._df['purchase_date'].max().date()
        return f'{mn} – {mx}'

    # ── Дандер-методи ──────────────────────────────────────────────────────

    def __repr__(self):
        """Читабельне подання — корисне в REPL і при дебагінгу."""
        return (f'DatasetAnalyzer(name={self._name!r}, '
                f'rows={self.row_count:,}, dates={self.date_range})')

    def __len__(self):
        """Підтримує len(analyzer) — інтеграція з Python runtime."""
        return self.row_count

    def __contains__(self, user_id):
        """Підтримує 'U123' in analyzer — перевірка наявності покупця."""
        return user_id in self._df['user_id'].values

    # ── Приватний метод з кешем ────────────────────────────────────────────

    def _compute_metric(self, key, func):
        """Обчислює метрику один раз, далі повертає з кешу."""
        if key not in self._cache:
            self._cache[key] = func()
        return self._cache[key]

    # ── Публічні метрики ───────────────────────────────────────────────────

    def avg_order_value(self):
        return self._compute_metric(
            'aov', lambda: round(self.delivered['final_price'].mean(), 2)
        )

    def return_rate(self):
        return self._compute_metric(
            'rr', lambda: round(self._df['is_returned'].mean(), 4)
        )

    def total_revenue(self):
        return self._compute_metric(
            'rev', lambda: round(self.delivered['final_price'].sum(), 2)
        )

    def summary(self):
        print(f'=== {self._name} ===')
        print(f'  Записів:          {len(self):>10,}')
        print(f'  Дати:              {self.date_range}')
        print(f'  Виручка:          ₹{self.total_revenue():>12,.0f}')
        print(f'  Середній чек:     ₹{self.avg_order_value():>12,.2f}')
        print(f'  Ставка повернень:  {self.return_rate():>10.1%}')


# Демонстрація всіх можливостей
a = DatasetAnalyzer(df, name='Amazon India')

print(repr(a))                           # __repr__
print(f'len(a) = {len(a):,}')           # __len__
print(f'row_count = {a.row_count:,}')   # @property
print(f'date_range = {a.date_range}')   # @property

# Перевіряємо, чи є конкретний покупець
first_user = df['user_id'].iloc[0]
print(f'{first_user!r} in analyzer: {first_user in a}')
print()
a.summary()

## Частина 7 — Наслідування: Спеціалізовані Аналізатори

---



### Проблема: один клас, який робить все = «God Object»

Якщо ми додамо в `DatasetAnalyzer` методи для продавців, регіонів, брендів —
клас стане незрозумілим і складним для підтримки.

**Рішення — наслідування як спеціалізація:**

```
DatasetAnalyzer          ← базова логіка (метрики, кеш, repr)
    ├── SalesAnalyzer    ← продажі по категоріях, брендах, локаціях
    ├── ReturnAnalyzer   ← аналіз повернень (причини, динаміка)
    └── SellerAnalyzer   ← аналіз продавців (рейтинг, виручка)
```

> `SalesAnalyzer` **є** `DatasetAnalyzer` (is-a) → наслідування виправдане.
> Підклас успадковує базову логіку і **додає тільки своє**.

In [ ]:
# SalesAnalyzer — аналіз продажів по категоріях, брендах, містах

class SalesAnalyzer(DatasetAnalyzer):
    """Спеціалізований аналіз продажів."""

    def __init__(self, df, name='Sales Analysis'):
        super().__init__(df, name)  # делегуємо батьківській __init__

    def revenue_by_category(self):
        """Виручка по категоріях (Delivered), відсортована."""
        return (
            self.delivered
            .groupby('category')['final_price']
            .sum()
            .sort_values(ascending=False)
            .round(0)
        )

    def avg_discount_by_category(self):
        """Середня знижка по категоріях."""
        return (
            self._df
            .groupby('category')['discount']
            .mean()
            .sort_values(ascending=False)
            .round(2)
        )

    def top_locations_by_orders(self, n=5):
        """Топ-N міст за кількістю замовлень."""
        return self._df['location'].value_counts().head(n)

    def rating_vs_discount(self):
        """Кореляція між знижкою і оцінкою покупця."""
        corr = self._df[['discount', 'rating']].corr().iloc[0, 1]
        return round(corr, 4)

    def summary(self):
        super().summary()  # виводимо базові KPI від батька
        print('\n  Виручка по категоріях:')
        for cat, rev in self.revenue_by_category().items():
            print(f'    {cat:<15} ₹{rev:>12,.0f}')
        print(f'\n  Корел. знижка-рейтинг: {self.rating_vs_discount()}')


sales = SalesAnalyzer(df)
sales.summary()
print('\nТоп-5 міст:')
print(sales.top_locations_by_orders())

In [ ]:
# ReturnAnalyzer — аналіз повернень: по категоріях, брендах, доставці

class ReturnAnalyzer(DatasetAnalyzer):
    """Аналіз повернень: що повертають і чому."""

    def __init__(self, df, name='Return Analysis'):
        super().__init__(df, name)

    def return_rate_by_category(self):
        """Ставка повернень по категоріях."""
        total = self._df.groupby('category').size()
        returned = self._df[self._df['is_returned']].groupby('category').size()
        rate = (returned / total).fillna(0).sort_values(ascending=False)
        return rate.round(4)

    def return_rate_by_brand(self, top_n=10):
        """Топ-N брендів з найвищою ставкою повернень (мін 50 замовлень)."""
        total = self._df.groupby('brand').size()
        # Фільтруємо бренди з достатньою кількістю замовлень
        popular_brands = total[total >= 50].index
        filtered = self._df[self._df['brand'].isin(popular_brands)]
        total_pop = filtered.groupby('brand').size()
        returned_pop = filtered[filtered['is_returned']].groupby('brand').size()
        rate = (returned_pop / total_pop).fillna(0).nlargest(top_n)
        return rate.round(4)

    def late_delivery_return_correlation(self):
        """Чи впливає затримка доставки на повернення?"""
        corr = self._df[['shipping_time_days', 'is_returned']].corr().iloc[0, 1]
        return round(corr, 4)

    def summary(self):
        super().summary()
        print('\n  Ставки повернень по категоріях:')
        for cat, rate in self.return_rate_by_category().items():
            print(f'    {cat:<15} {rate:.1%}')
        print(f'\n  Корел. затримка доставки → повернення: {self.late_delivery_return_correlation()}')


ret = ReturnAnalyzer(df)
ret.summary()

In [ ]:
# SellerAnalyzer — аналіз продавців: рейтинг, виручка, надійність

class SellerAnalyzer(DatasetAnalyzer):
    """Аналіз продавців: хто продає найбільше і найкраще."""

    def __init__(self, df, name='Seller Analysis'):
        super().__init__(df, name)

    def top_sellers_by_revenue(self, n=10):
        """Топ-N продавців за виручкою."""
        return (
            self.delivered
            .groupby('seller_id')['final_price']
            .sum()
            .nlargest(n)
            .round(0)
        )

    def seller_scorecard(self):
        """
        Зведена таблиця: виручка + ставка повернень + середній рейтинг.
        Відсортовано за виручкою.
        """
        revenue = self.delivered.groupby('seller_id')['final_price'].sum().round(0)
        total = self._df.groupby('seller_id').size()
        returned = self._df[self._df['is_returned']].groupby('seller_id').size()
        avg_rating = self._df.groupby('seller_id')['rating'].mean().round(2)

        scorecard = pd.DataFrame({
            'revenue':     revenue,
            'return_rate': (returned / total).fillna(0).round(3),
            'avg_rating':  avg_rating,
        }).sort_values('revenue', ascending=False)
        return scorecard

    def __repr__(self):
        n_sellers = self._df['seller_id'].nunique()
        return f'SellerAnalyzer(sellers={n_sellers:,}, rows={self.row_count:,})'


seller = SellerAnalyzer(df)
print(repr(seller))
print('\nТоп-5 продавців за виручкою:')
print(seller.top_sellers_by_revenue(5))
print('\nScorecard (перші 5):')
print(seller.seller_scorecard().head())

## Частина 8 — Композиція: Збираємо Аналітичний Рушій

---



### Чому НЕ робимо `class Engine(Sales, Returns, Sellers)`?

Множинне наслідування для цього випадку — це антипаттерн:
- Клас отримує **всі** методи трьох класів одночасно → God Object знову
- Конфлікти імен методів → неочевидна поведінка через MRO
- Важко тестувати: не зрозуміло, що за що відповідає

**Правильно:** патерн **Facade через Композицію (Has-a)**:

```
AnalyticsEngine
    ├── .sales   → SalesAnalyzer    делегуємо питання продажів
    ├── .returns → ReturnAnalyzer   делегуємо питання повернень
    └── .sellers → SellerAnalyzer   делегуємо питання продавців
```

`AnalyticsEngine` — **єдина точка входу** до всієї системи аналітики.

In [ ]:
class AnalyticsEngine:
    """
    Головний аналітичний рушій — Facade Pattern через композицію.
    Не наслідує жоден аналізатор — замість цього МІСТИТЬ їх.
    """

    def __init__(self, df: pd.DataFrame, name: str = 'E-commerce Analytics'):
        self.name = name
        # Кожен аналізатор незалежний; отримує той самий df
        self.sales   = SalesAnalyzer(df,  name=f'{name}/Sales')
        self.returns = ReturnAnalyzer(df, name=f'{name}/Returns')
        self.sellers = SellerAnalyzer(df, name=f'{name}/Sellers')

    def full_report(self):
        """Повний звіт — делегуємо кожному аналізатору своє."""
        print('=' * 55)
        print(f'  {self.name}')
        print('=' * 55)
        self.sales.summary()
        print()
        print('  Ставки повернень по категоріях:')
        for cat, rate in self.returns.return_rate_by_category().items():
            print(f'    {cat:<15} {rate:.1%}')
        print()
        print('  Топ-5 продавців:')
        for sid, rev in self.sellers.top_sellers_by_revenue(5).items():
            print(f'    {sid:<12} ₹{rev:>12,.0f}')
        print('=' * 55)

    def kpi_dict(self) -> dict:
        """Словник KPI — зручний формат для дашбордів і API."""
        return {
            'total_revenue':         self.sales.total_revenue(),
            'avg_order_value':       self.sales.avg_order_value(),
            'return_rate':           self.returns.return_rate(),
            'top_category':          self.sales.revenue_by_category().index[0],
            'top_seller':            self.sellers.top_sellers_by_revenue(1).index[0],
            'late_return_corr':      self.returns.late_delivery_return_correlation(),
        }


# Запускаємо рушій
engine = AnalyticsEngine(df, name='Amazon India — Analytics')
engine.full_report()

print()
print('KPI dict (для дашборду):')
for k, v in engine.kpi_dict().items():
    print(f'  {k:<25} {v}')

## Частина 9 — Контекстний Менеджер: `DataLoader`

---



### Навіщо тут контекстний менеджер?

Аналіз великих файлів — це IO: відкриваємо CSV (127 MB!), читаємо, аналізуємо.
Без контекстного менеджера: якщо під час читання стається помилка —
файловий дескриптор залишається відкритим → **витік ресурсів**.

Контекстний менеджер `DataLoader`:
- `__enter__` → відкриває файл, читає CSV, будує `AnalyticsEngine`
- `__exit__` → логує завершення, може зберегти звіт у файл

```python
with DataLoader('amazon_ecommerce_1M.csv') as engine:
    engine.full_report()
# Тут ресурси гарантовано вивільнені — навіть якщо full_report() впав
```

In [ ]:
from contextlib import contextmanager


class DataLoader:
    """
    Контекстний менеджер: завантажує CSV і повертає AnalyticsEngine.
    __enter__ → читає CSV → будує Engine → повертає його
    __exit__  → логує завершення, може зберегти кеш метрик
    """

    def __init__(self, filepath: str, name: str = 'Analytics', nrows=None):
        self._filepath = Path(filepath)
        self._name = name
        self._nrows = nrows  # None = всі рядки
        self._engine = None

    def __enter__(self):
        print(f'[DataLoader] Відкриваємо: {self._filepath}')
        df = pd.read_csv(
            self._filepath,
            parse_dates=['purchase_date'],
            nrows=self._nrows
        )
        print(f'[DataLoader] Завантажено: {len(df):,} рядків')
        self._engine = AnalyticsEngine(df, name=self._name)
        return self._engine  # → присвоюється змінній після 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f'[DataLoader] Помилка: {exc_type.__name__}: {exc_val}')
        else:
            print('[DataLoader] Аналіз завершено успішно')
        print('[DataLoader] Ресурси вивільнено\n')
        return False  # не пригнічуємо виняток


# Варіант 1: клас
print('=== DataLoader (клас) ===')
with DataLoader('amazon_ecommerce_1M.csv', name='Amazon India', nrows=200_000) as engine:
    kpi = engine.kpi_dict()

print('KPI:')
for k, v in kpi.items():
    print(f'  {k:<25} {v}')


# Варіант 2: @contextmanager (лаконічніший)
@contextmanager
def load_dataset(filepath, name='Analytics', nrows=None):
    """Варіант через генератор — менше коду, та сама гарантія."""
    print(f'[load_dataset] Читаємо: {filepath}')
    df = pd.read_csv(filepath, parse_dates=['purchase_date'], nrows=nrows)
    engine = AnalyticsEngine(df, name=name)
    try:
        yield engine
    finally:
        print('[load_dataset] Завершено, ресурси вивільнено')


print()
print('=== load_dataset (@contextmanager) ===')
with load_dataset('amazon_ecommerce_1M.csv', nrows=100_000) as eng:
    print(f'  Виручка: ₹{eng.kpi_dict()["total_revenue"]:,.0f}')

## Частина 10 — Streamlit: Мінімальний Дашборд

---

### Ідея: наш `AnalyticsEngine` + Streamlit = продукт за 30 рядків

Streamlit перетворює Python-скрипт на веб-застосунок.

**Запуск:**
```bash
cd module_2/lessons/lesson_21_module2_review/ecommerce_analytics
  streamlit run app.py
```

Дашборд покаже:
- Загальна статистика
- Фільтр по категорії у боковій панелі
- Bar chart: Кількість пропозицій за категоріями
- Bar chart: Середня ціна за категоріями
- Bar chart: Топ-10 брендів за кількістю пропозицій
- Порівняння категорій: рейтинг та ціна


## Підсумок: Від Функцій до Системи

---

### Архітектура того, що ми побудували

```
amazon_ecommerce_1M.csv  (1M рядків)
          │
          ▼
   DataLoader  (контекстний менеджер)
          │  __enter__ → pd.read_csv() → AnalyticsEngine
          ▼
   AnalyticsEngine  ◄── Facade Pattern (композиція)
     ├── SalesAnalyzer(DatasetAnalyzer)
     │       revenue_by_category(), rating_vs_discount()
     ├── ReturnAnalyzer(DatasetAnalyzer)
     │       return_rate_by_category(), late_delivery_correlation()
     └── SellerAnalyzer(DatasetAnalyzer)
             top_sellers_by_revenue(), seller_scorecard()
          │
          ▼
     dashboard.py  →  Streamlit UI
```

---

### Еволюція коду: від хаосу до системи

| Крок | Інструмент Python | Навіщо |
|------|-------------------|--------|
| 1 | Прості функції | Ізольовані, тестовані метрики |
| 2 | Функції у списку | Передаємо функції як аргументи |
| 3 | Пайплайн функцій | `[f1, f2, f3]` — декларативна обробка |
| 4 | Декоратори | Логування і таймінг без дублювання |
| 5 | Клас (v1) | Стан + методи в одному просторі імен |
| 6 | `@property`, `__repr__`, `__len__` | Клас «говорить» мовою Python runtime |
| 7 | Наслідування | Спеціалізація без дублювання базової логіки |
| 8 | Композиція (Facade) | Один об'єкт координує кількох спеціалістів |
| 9 | Контекстний менеджер | Гарантоване управління ресурсами |
| 10 | Streamlit | Дані → інтерактивна веб-сторінка |

---

**Головний урок Module 2:**
- Програмування — це управління складністю.
- Функції → Класи → Патерни — це не ускладнення.
Це інструменти, що дозволяють **системі рости без хаосу**.

> - Просте краще за складне.
> - Архітектура — це спосіб залишити систему простою.